# Exploratory Data Analysis: Credit Card Fraud Detection

This notebook performs exploratory data analysis on the Kaggle Credit Card Fraud Detection dataset.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

In [ ]:
# Load data
data_path = '../data/raw/creditcard.csv'
df = pd.read_csv(data_path)
print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
# Basic statistics
print("Basic statistics:")
df.describe()

In [ ]:
# Class distribution
plt.figure(figsize=(8, 6))
class_counts = df['Class'].value_counts()
sns.barplot(x=class_counts.index, y=class_counts.values)
plt.title('Class Distribution')
plt.xlabel('Class (0=Legit, 1=Fraud)')
plt.ylabel('Count')
plt.yscale('log')
plt.show()

print(f"Fraud ratio: {class_counts[1] / len(df):.6f}")

In [ ]:
# Time distribution
plt.figure(figsize=(12, 6))
plt.hist(df['Time'], bins=50, alpha=0.7)
plt.title('Transaction Time Distribution')
plt.xlabel('Time (seconds)')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Amount distribution
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.hist(df['Amount'], bins=50, alpha=0.7)
plt.title('Transaction Amount Distribution')
plt.xlabel('Amount')
plt.ylabel('Frequency')

plt.subplot(1, 2, 2)
plt.hist(df[df['Amount'] < 100]['Amount'], bins=50, alpha=0.7)
plt.title('Transaction Amount Distribution (< $100)')
plt.xlabel('Amount')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

In [ ]:
# Feature distributions (V1-V28)
fig, axes = plt.subplots(7, 4, figsize=(20, 25))
axes = axes.flatten()

for i, col in enumerate([f'V{j}' for j in range(1, 29)]):
    axes[i].hist(df[col], bins=50, alpha=0.7)
    axes[i].set_title(f'{col} Distribution')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
corr_matrix = df.corr()
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, annot=False)
plt.title('Feature Correlation Matrix')
plt.show()

In [ ]:
# Correlation with Class
correlations = df.corr()['Class'].sort_values(ascending=False)
plt.figure(figsize=(10, 8))
correlations.drop('Class').plot(kind='barh')
plt.title('Feature Correlations with Class')
plt.xlabel('Correlation')
plt.show()

In [ ]:
# Temporal splitting simulation
df_sorted = df.sort_values('Time').reset_index(drop=True)
n_samples = len(df_sorted)
era_size = n_samples // 4

eras = []
for i in range(4):
    start_idx = i * era_size
    end_idx = (i + 1) * era_size if i < 3 else n_samples
    era_df = df_sorted.iloc[start_idx:end_idx]
    eras.append(era_df)

# Class distribution per era
for i, era in enumerate(eras):
    class_dist = era['Class'].value_counts()
    print(f"Era {i}: Legit={class_dist.get(0, 0)}, Fraud={class_dist.get(1, 0)}, Ratio={class_dist.get(1, 0)/len(era):.6f}")

In [ ]:
# Amount normalization
scaler = StandardScaler()
scaler.fit(eras[0][['Amount']])

for i in range(4):
    eras[i] = eras[i].copy()
    eras[i]['Amount_norm'] = scaler.transform(eras[i][['Amount']])

# Plot normalized Amount per era
plt.figure(figsize=(12, 8))
for i in range(4):
    plt.subplot(2, 2, i+1)
    plt.hist(eras[i]['Amount_norm'], bins=50, alpha=0.7)
    plt.title(f'Era {i} Normalized Amount')
    plt.xlabel('Normalized Amount')
    plt.ylabel('Frequency')
plt.tight_layout()
plt.show()